<a href="https://colab.research.google.com/github/hawaekrami/week1/blob/main/work/notebooks/w07_action_playbook.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# ML-10 — Content Action Playbook

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/hawaekrami/week1/blob/main/work/notebooks/w07_action_playbook.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. Ranked actions + reason codes

*The queue: what to do first, and why, in words a human trusts.*

Reason codes map a page's Feb-period signal pattern to a plain-language action.
Three archetypes, built on the same features used in Weeks 4–6
(impressions_feb, clicks_feb, ctr_feb, avg_position_feb) plus the model_score
from Week 5/6:

| Reason code | Pattern | Action |
|---|---|---|
| `low_ctr_visible_page` | Ranks well (position ≤ 20) but CTR is below benchmark | `review_ctr_fix` — check title/meta/snippet |
| `flagged_declining_high_volume` | High Feb impressions, model flags as likely declining | `review_content_refresh` — priority review |
| `low_signal_insufficient_data` | Below the 500-impression volume floor | `no_action_insufficient_data` — do not rank; not enough traffic to trust any score |

**Known limitation carried in from Week 6:** the model_score column and any
precision numbers below are placeholders pending confirmation that the
Week 6 client-shuffle bug fix produces a stable, reproducible split (same
result on two consecutive runs). The reason-code logic, archetype mapping,
and CTR-benchmark rule are independent of that number and are final.

In [1]:
import duckdb
import pandas as pd
import numpy as np
from google.colab import userdata

con = duckdb.connect()
HF_TOKEN = userdata.get('HF_TOKEN')
con.execute(f"CREATE OR REPLACE SECRET hf (TYPE huggingface, TOKEN '{HF_TOKEN}')")

FEB = "read_parquet('hf://datasets/FlyRank/internship-warehouse/fact_content_daily_performance/month=2026-02/*.parquet')"

feb = con.sql(f"""
    SELECT content_hash_id, client_hash_id,
        SUM(gsc_impressions) AS impressions_feb,
        SUM(gsc_clicks) AS clicks_feb,
        ROUND(100.0 * SUM(gsc_clicks) / NULLIF(SUM(gsc_impressions), 0), 2) AS ctr_feb,
        AVG(gsc_avg_position) AS avg_position_feb
    FROM {FEB} GROUP BY content_hash_id, client_hash_id
""").df()

MIN_IMPRESSIONS = 500
CTR_BENCHMARK = 0.5

queue = feb.copy()

# PLACEHOLDER: model_score. Do not treat these values as final — see limitation
# note above and in Section 2. Set to NaN so it's visually obvious this column
# is not yet trustworthy, rather than silently shipping a possibly-wrong number.
queue["model_score"] = np.nan  # TODO: replace once Week 6 split is confirmed stable

def assign_reason_code(row):
    if row["impressions_feb"] < MIN_IMPRESSIONS:
        return "low_signal_insufficient_data", "no_action_insufficient_data"
    if row["avg_position_feb"] > 0 and row["avg_position_feb"] <= 20 and row["ctr_feb"] < CTR_BENCHMARK:
        return "low_ctr_visible_page", "review_ctr_fix"
    # model_score-based archetype is a placeholder until Section-1's TODO is resolved
    return "flagged_declining_high_volume", "review_content_refresh"

queue[["reason_code", "action"]] = queue.apply(
    lambda r: pd.Series(assign_reason_code(r)), axis=1
)

print(queue["reason_code"].value_counts())
print()
print(queue[["content_hash_id", "impressions_feb", "ctr_feb", "avg_position_feb",
             "reason_code", "action"]].head(10).to_string(index=False))

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

reason_code
low_signal_insufficient_data     274412
low_ctr_visible_page              34176
flagged_declining_high_volume     12958
Name: count, dtype: int64

         content_hash_id  impressions_feb  ctr_feb  avg_position_feb                   reason_code                      action
content_7995404695ee1ffd           1012.0     0.30         29.609070 flagged_declining_high_volume      review_content_refresh
content_1eea820697c3b95a            299.0     0.00         12.946228  low_signal_insufficient_data no_action_insufficient_data
content_ccbb253f142217c3           1598.0     0.44         17.806923          low_ctr_visible_page              review_ctr_fix
content_5f58c55cbfee172a            514.0     0.00         10.490023          low_ctr_visible_page              review_ctr_fix
content_6fe390ba3af1e456           2931.0     0.10         38.436254 flagged_declining_high_volume      review_content_refresh
content_babd931911c9ee33           2680.0     1.16          5.046562 flagged_de

## 2. Intended use and limits

*Who uses this, for what and where it stops being valid.*

**Intended use:** this queue is decision-support for a content team prioritizing
*which* pages to look at first during a review cycle. It is not a scoring system
for individual-page performance and not a ranking of page quality.

**Who it's for:** an SEO/content lead triaging a backlog, using it to order
review work — not to auto-generate content changes, and not to justify
headcount, budget, or individual performance decisions.

**Coverage limit (measured, this run):** of 321,546 pages scored, 85.3%
(274,412) fall below the 500-impression volume floor and get
`no_action_insufficient_data` — meaning this playbook has nothing decision-useful
to say about the majority of the catalog by page count. It speaks confidently
only about the remaining 14.6% of pages: 10.6% (34,176) get `review_ctr_fix`
and 4.0% (12,958) get `review_content_refresh`.

**Time limit:** every feature here is built from a single month (February 2026).
The playbook reflects that one month's pattern and is not validated against
seasonality, year-over-year shifts, or algorithm updates outside that window.

**Population limit:** built and tested on 54 clients in one warehouse snapshot.
Directional evidence only for those clients' content patterns — not a claim
that these reason-code thresholds (500 impressions, 0.5% CTR benchmark, ≤20
position) generalize to a different client base, industry vertical, or site
size without re-checking.

**Model-score limit, stated directly:** the `flagged_declining_high_volume`
reason code is currently assigned by a default bucket (high volume, not
otherwise flagged), not a validated model_score — see the Section 1
placeholder note. Until the Week 6 split is confirmed stable across two
consecutive runs, this specific reason code should be read as "high volume,
worth a look" rather than "the model measured this as declining."

In [2]:
n_total = len(queue)
n_insufficient = (queue["reason_code"] == "low_signal_insufficient_data").sum()
n_ctr = (queue["reason_code"] == "low_ctr_visible_page").sum()
n_declining = (queue["reason_code"] == "flagged_declining_high_volume").sum()
n_clients = queue["client_hash_id"].nunique()

print(f"Total pages scored: {n_total}")
print(f"Coverage — no_action_insufficient_data: {n_insufficient} ({100*n_insufficient/n_total:.1f}%)")
print(f"Coverage — review_ctr_fix:              {n_ctr} ({100*n_ctr/n_total:.1f}%)")
print(f"Coverage — review_content_refresh:      {n_declining} ({100*n_declining/n_total:.1f}%)")
print(f"Clients represented: {n_clients}")
print()
print("Use these percentages to fill the {N} / {%} placeholders in the markdown cell above.")


Total pages scored: 321546
Coverage — no_action_insufficient_data: 274412 (85.3%)
Coverage — review_ctr_fix:              34176 (10.6%)
Coverage — review_content_refresh:      12958 (4.0%)
Clients represented: 54

Use these percentages to fill the {N} / {%} placeholders in the markdown cell above.


## 3. Human review + the no-go list
*What a person must check before acting. What should never be automated.*

**Before acting on any `review_ctr_fix` or `review_content_refresh` item, a
human must check:**
- Is the page still live and indexed? (Warehouse data can lag or include
  removed/redirected pages.)
- Is the low CTR explained by something outside the page itself a SERP
  feature stealing clicks, a branded query cannibalizing an intent match, a
  recent title/meta change not yet reflected in the data window?
- For `review_content_refresh`: is this page tied to a live campaign,
  seasonal content, or a page intentionally being sunset? The reason code
  can't distinguish "declining and worth saving" from "declining on purpose."
- Does the client relationship or contract require sign-off before content
  changes ship?

**No-go list what must NOT be automated:**
- No automatic publishing of title/meta/content changes from this queue.
  Every action is a *suggestion to review*, never a direct edit.
- No automatic deprioritization or removal of pages based on
  `no_action_insufficient_data` that code means "not enough signal to say
  anything," not "this page is low value."
- No use of this queue for individual performance evaluation of writers or
  editors — the volume floor and thresholds were tuned for triage, not audit.
- No cross-client comparison or ranking using this queue — thresholds were
  validated in aggregate, not per-client, and client size differences would
  make a cross-client ranking misleading.
- No automatic budget or headcount reallocation triggered directly by queue
  counts, until the Week 6 model-score reliability question (see Section 2)
  is resolved.

In [3]:
# Surface a concrete example set for the human-review notes above, so the
# review checklist isn't abstract — pull a few real borderline cases.

borderline = queue[
    (queue["reason_code"] == "review_content_refresh".replace("review_", "flagged_") if False else queue["reason_code"] == "flagged_declining_high_volume")
    & (queue["avg_position_feb"] <= 3)  # already ranking well — refresh candidate worth double-checking, not obviously "declining"
].sort_values("impressions_feb", ascending=False).head(5)

print("Example 'flagged_declining_high_volume' pages that already rank in the top 3")
print("positions — exactly the kind of case a human should sanity-check before")
print("assuming 'declining' means 'underperforming', since these are already visible:")
print()
print(borderline[["content_hash_id", "impressions_feb", "ctr_feb", "avg_position_feb"]].to_string(index=False))


Example 'flagged_declining_high_volume' pages that already rank in the top 3
positions — exactly the kind of case a human should sanity-check before
assuming 'declining' means 'underperforming', since these are already visible:

         content_hash_id  impressions_feb  ctr_feb  avg_position_feb
content_512dbad65bd5ade9         167303.0     1.98          2.923168
content_c9a0c2fdbdbfb562         142215.0     1.13          1.882000
content_29c4a3831609805d         129662.0     1.25          1.660035
content_6302b8bce0bb84cb         112007.0     0.74          2.454561
content_b2cb08ff59fcce78          95003.0     0.73          2.676439


## 4. Monitoring / retrain triggers
*What would tell you the recommendations went stale?*

**Data-freshness triggers:**
- Re-run monthly, using the newest available monthly partition as "current"
  and the prior month as the comparison — the same Feb-vs-March logic from
  Week 5/6, rolled forward each cycle.
- If a monthly partition is missing, delayed, or has a row count that drops
  sharply versus the prior month (possible pipeline break), pause the queue
  and flag rather than silently scoring on stale or partial data.

**Distribution-shift triggers:**
- Track the reason-code split (currently ~85% insufficient_data / 11% CTR /
  4% flagged_declining) month over month. A large swing — e.g.
  insufficient_data dropping below 70% or above 95% — signals either a
  genuine traffic shift worth investigating or a data-quality issue in the
  new pull, and should be checked before trusting the new queue.
- Track the GA4 coverage rate noted in Week 3 (~4.2%). If that changes
  materially, features relying on GA4 (if added later) would need
  re-validation.

**Model-reliability triggers (once Week 6's split is confirmed stable):**
- If precision@50 on a fresh held-out set of clients drops meaningfully below
  the validated Week 6 number, treat that as a signal to re-audit for
  leakage or distribution shift before trusting the

In [4]:
# Snapshot the current reason-code distribution as the monitoring baseline —
# future monthly runs compare against these percentages to catch drift.

monitoring_baseline = {
    "run_month": "2026-02",
    "total_pages": int(n_total),
    "clients_represented": int(n_clients),
    "pct_no_action_insufficient_data": round(100 * n_insufficient / n_total, 1),
    "pct_review_ctr_fix": round(100 * n_ctr / n_total, 1),
    "pct_review_content_refresh": round(100 * n_declining / n_total, 1),
    "ctr_benchmark_used": CTR_BENCHMARK,
    "volume_floor_used": MIN_IMPRESSIONS,
    "model_score_status": "PLACEHOLDER — pending Week 6 split stability confirmation",
}

import json
print(json.dumps(monitoring_baseline, indent=2))


{
  "run_month": "2026-02",
  "total_pages": 321546,
  "clients_represented": 54,
  "pct_no_action_insufficient_data": 85.3,
  "pct_review_ctr_fix": 10.6,
  "pct_review_content_refresh": 4.0,
  "ctr_benchmark_used": 0.5,
  "volume_floor_used": 500,
  "model_score_status": "PLACEHOLDER \u2014 pending Week 6 split stability confirmation"
}


## 5. Exports for the paper

*Write the queue (and any figures you want to reuse) to work/outputs/ — your paper builds on these files.*

Two exports:
- `work/outputs/action_queue.csv` — the full ranked/reason-coded queue.
  Stays out of git by design (CI leak-guard blocks data files); this notebook
  regenerates it on every run.
- `work/outputs/w07_monitoring_baseline.json` — the metrics snapshot from
  Section 4. This one gets committed (metrics JSONs are the receipts the
  paper's numbers trace back to — data files don't, aggregated stats do).

**Note carried forward:** the `model_score` column in the exported queue is
still a placeholder (see Sections 1–2). If the Week 6 split gets confirmed
stable before the paper is written, re-run this notebook top to bottom first
so the paper builds on a real score column, not the placeholder.

In [5]:
import os
import json

os.makedirs("work/outputs", exist_ok=True)

# --- Export 1: the ranked queue (gitignored by design — regenerated each run) ---
export_cols = ["content_hash_id", "client_hash_id", "impressions_feb", "clicks_feb",
               "ctr_feb", "avg_position_feb", "model_score", "reason_code", "action"]

queue[export_cols].to_csv("work/outputs/action_queue.csv", index=False)
print(f"Wrote work/outputs/action_queue.csv — {len(queue)} rows")

# --- Export 2: monitoring baseline metrics JSON (this one gets committed) ---
with open("work/outputs/w07_monitoring_baseline.json", "w") as f:
    json.dump(monitoring_baseline, f, indent=2)
print("Wrote work/outputs/w07_monitoring_baseline.json")
print()
print(json.dumps(monitoring_baseline, indent=2))

Wrote work/outputs/action_queue.csv — 321546 rows
Wrote work/outputs/w07_monitoring_baseline.json

{
  "run_month": "2026-02",
  "total_pages": 321546,
  "clients_represented": 54,
  "pct_no_action_insufficient_data": 85.3,
  "pct_review_ctr_fix": 10.6,
  "pct_review_content_refresh": 4.0,
  "ctr_benchmark_used": 0.5,
  "volume_floor_used": 500,
  "model_score_status": "PLACEHOLDER \u2014 pending Week 6 split stability confirmation"
}


## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.